In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# ==========================================
# 3. إعداد وتنظيف البيانات للتحليل (المطلب الثالث)
# ==========================================
# تحميل البيانات
df = pd.read_excel('/content/Shady file.xlsx') # Changed to pd.read_excel

# تحويل الأعمدة المالية إلى Float لتجنب مشاكل الأرقام العشرية
numeric_cols = ['Price', 'Quantity', 'Order Total'] # Example columns, adjust as needed
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# الاستيفاء الرياضي الصارم (Financial Imputation)
# These imputation steps need to be carefully reviewed as they use df directly for fillna values,
# which might not be intended. It seems to imply df['Price'] * df['Quantity'] for missing values in df itself,
# which is syntactically incorrect for fillna. Assuming the intention is to fill NaNs in 'Order Total'.
df['Order Total'] = df['Order Total'].fillna(df['Price'] * df['Quantity'])
df['Price'] = df['Price'].fillna(df['Order Total'] / df['Quantity'])
df['Quantity'] = df['Quantity'].fillna(df['Order Total'] / df['Price'])

# معالجة الأسعار المتبقية باستخدام المنوال (Mode)
def get_price_mode(series):
    modes = series.mode()
    return modes.iloc[0] if not modes.empty else np.nan # Added .iloc[0] to get a single mode value
df['Price'] = df['Price'].fillna(df.groupby('Item')['Price'].transform(get_price_mode))

# الاستيفاء المنطقي للأصناف المفقودة
def get_item_mode(series):
    modes = series.mode()
    return modes.iloc[0] if not modes.empty else 'Unknown' # Changed np.nan to 'Unknown' for categorical data consistency
df['Item'] = df['Item'].fillna(df.groupby(['Category', 'Price'])['Item'].transform(get_item_mode))
df['Item'] = df['Item'].fillna('Unknown')

# الاستيفاء الفئوي للمدفوعات
df['Payment Method'] = df['Payment Method'].fillna('Unknown')

# حذف الصفوف التالفة بالكامل والنسخ المكررة
critical_columns = ['Order ID', 'Customer ID', 'Item', 'Category', 'Price', 'Quantity', 'Order Total', 'Order Date', 'Payment Method'] # Example columns, adjust as needed
df = df.dropna(subset=critical_columns, how='all')
df = df.drop_duplicates()

# ضبط صيغة التواريخ
# Assuming 'Order Date' is the column containing dates
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')

# ==========================================
# 1. تصدير ملف الإكسيل النظيف (المطلب الأول)
# ==========================================
df.to_excel('Shady file cleaned.xlsx', index=False)

# ==========================================
# 2. إنشاء وتصدير لوحة بيانات HTML (المطلب الثاني)
# ==========================================
# حساب المؤشرات
total_revenue = df['Order Total'].sum() # Specify column for sum
total_orders = df['Order ID'].nunique() # Specify column for unique count
avg_order_value = total_revenue / total_orders if total_orders > 0 else 0

# ألوان Power BI
pb_colors = ['#01B8AA', '#374649', '#FD625E', '#F2C80F', '#5F6B6F', '#A3CEDC'] # Example colors

# الرسوم البيانية
top_items = df.groupby('Item')['Quantity'].sum().reset_index().sort_values(by='Quantity').tail(10)
fig_items = px.bar(top_items, x='Quantity', y='Item', orientation='h', title="Top Selling Items", color_discrete_sequence=pb_colors)
fig_items.update_layout(plot_bgcolor='white', paper_bgcolor='white', margin=dict(t=40, l=10, r=10, b=10))

cat_rev = df.groupby('Category')['Order Total'].sum().reset_index() # Specify column for sum
fig_cat = px.pie(cat_rev, values='Order Total', names='Category', hole=0.6, title="Revenue by Category", color_discrete_sequence=pb_colors)
fig_cat.update_layout(plot_bgcolor='white', paper_bgcolor='white', margin=dict(t=40, l=10, r=10, b=10))

pay_ord = df.groupby('Payment Method')['Order ID'].nunique().reset_index().sort_values(by='Order ID', ascending=False) # Specify column for unique count
fig_pay = px.bar(pay_ord, x='Payment Method', y='Order ID', title="Orders by Payment Method", color_discrete_sequence=['#374649'])
fig_pay.update_layout(plot_bgcolor='white', paper_bgcolor='white', margin=dict(t=40, l=10, r=10, b=10))

monthly_rev = df.copy()
# Ensure 'Order Date' exists and is datetime type before accessing .dt
if 'Order Date' in monthly_rev.columns and pd.api.types.is_datetime64_any_dtype(monthly_rev['Order Date']):
    monthly_rev['Month'] = monthly_rev['Order Date'].dt.to_period('M').astype(str)
else:
    monthly_rev['Month'] = 'Unknown' # Handle cases where 'Order Date' is missing or not datetime

monthly_rev = monthly_rev.groupby('Month')['Order Total'].sum().reset_index() # Specify column for sum
fig_trend = px.line(monthly_rev, x='Month', y='Order Total', markers=True, title="Monthly Revenue Trend", color_discrete_sequence=pb_colors)
fig_trend.update_layout(plot_bgcolor='white', paper_bgcolor='white', margin=dict(t=40, l=10, r=10, b=10))

# Corrected creation of fig_table
top_cust_data = df.groupby('Customer ID').agg({'Order Total': 'sum', 'Order ID': 'nunique'}).reset_index().sort_values(by='Order Total', ascending=False).head(10)

# Prepare cell values, applying formatting only to 'Order Total' for display
cell_values = [
    top_cust_data['Customer ID'].tolist(),
    top_cust_data['Order Total'].apply(lambda x: f"${x:,.2f}").tolist(),
    top_cust_data['Order ID'].tolist()
]

fig_table = go.Figure(data=[
    go.Table(
        header=dict(values=list(top_cust_data.columns),
                    fill_color='#374649',
                    font=dict(color='white'),
                    align='left'),
        cells=dict(values=cell_values,
                   fill_color='#F2F3F8',
                   font=dict(color='#000000'),
                   align='left'))
])
fig_table.update_layout(title="Top 10 Customers", margin=dict(t=40, l=10, r=10, b=10))

# تجميع HTML
html_content = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Dashboard</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        body {{ font-family: 'Segoe UI', Tahoma, sans-serif; background-color: #F2F3F8; margin: 0; padding: 20px; }}
       .kpi-container {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(250px, 1fr)); gap: 20px; margin-bottom: 20px; }}
       .kpi-card {{ background: #FFF; border-radius: 6px; padding: 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.16); border-top: 4px solid #01B8AA; }}
       .kpi-card h3 {{ margin: 0 0 10px 0; font-size: 14px; color: #666; text-transform: uppercase; }}
       .kpi-card p {{ margin: 0; font-size: 28px; font-weight: bold; color: #374649; }}
       .grid-container {{ display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }}
       .chart-card {{ background: #FFF; border-radius: 6px; padding: 15px; box-shadow: 0 1px 3px rgba(0,0,0,0.16); height: 400px; }}
       .full-width {{ grid-column: span 2; }}
    </style>
</head>
<body>
    <h2 style="color:#374649; margin-bottom:20px;">Restaurant Sales Analytics</h2>
    <div class="kpi-container">
        <div class="kpi-card"><h3>Total Revenue</h3><p>${total_revenue:,.2f}</p></div>
        <div class="kpi-card" style="border-top-color:#374649;"><h3>Total Orders</h3><p>{total_orders:,}</p></div>
        <div class="kpi-card" style="border-top-color:#FD625E;"><h3>Avg Order Value</h3><p>${avg_order_value:,.2f}</p></div>
    </div>
    <div class="grid-container">
        <div class="chart-card full-width">{fig_trend.to_html(full_html=False, include_plotlyjs=False)}</div>
        <div class="chart-card">{fig_items.to_html(full_html=False, include_plotlyjs=False)}</div>
        <div class="chart-card">{fig_cat.to_html(full_html=False, include_plotlyjs=False)}</div>
        <div class="chart-card">{fig_pay.to_html(full_html=False, include_plotlyjs=False)}</div>
        <div class="chart-card">{fig_table.to_html(full_html=False, include_plotlyjs=False)}</div>
    </div>
</body>
</html>
"""

# تصدير الملف
with open("dashboard.html", "w", encoding="utf-8") as f:
    f.write(html_content)

print("تم بنجاح! الملفات 'Shady file cleaned.xlsx' و 'dashboard.html' جاهزة الآن للتحميل.")

تم بنجاح! الملفات 'Shady file cleaned.xlsx' و 'dashboard.html' جاهزة الآن للتحميل.
